# Notebook 02: Model Training
Swahili concern classification | Obonyo Paul Rapando (169688)

**Inputs** (Drive): `data/annotated/labelled_concerns.csv`, optional `resident_test.csv`
**Outputs**: `data/processed/` (split, embeddings), `models/` (classifiers), `results/training/` (metrics, metadata)

**Method**
1. Stratified 80/20 split. The test partition is never loaded here. Notebook 03 evaluates it.
2. AfroXLMR (frozen) turns each sentence into a 768-dimensional embedding using attention-mask mean pooling.
3. Logistic Regression, Linear SVM and XGBoost are tuned with the same 5-fold stratified CV on the training partition only.
4. Selection uses macro F1. SMOTE is used only if the training classes are imbalanced, and only inside each CV fold.

In [ ]:
%pip install -q imbalanced-learn

from google.colab import drive
import os
import re
import json
import time
import hashlib
from importlib.metadata import version

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import joblib
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModel
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, RandomizedSearchCV, cross_val_score
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

drive.mount('/content/drive')

PROJECT = '/content/drive/MyDrive/PaulObonyo_169688'
ANN_DIR = os.path.join(PROJECT, 'data', 'annotated')
PROC_DIR = os.path.join(PROJECT, 'data', 'processed')
MODELS_DIR = os.path.join(PROJECT, 'models')
RES_DIR = os.path.join(PROJECT, 'results', 'training')
for d in (PROC_DIR, MODELS_DIR, RES_DIR):
    os.makedirs(d, exist_ok=True)

SEED = 42
MODEL_NAME = 'Davlan/afro-xlmr-base'
BATCH_SIZE = 32
TEST_SIZE = 0.20
NORMALISE = True


def normalise(text):
    text = re.sub(r"[^\w\s]", ' ', str(text).lower())
    return re.sub(r'\s+', ' ', text).strip()


def file_md5(path):
    with open(path, 'rb') as f:
        return hashlib.md5(f.read()).hexdigest()


def pkg_version(name):
    try:
        return version(name)
    except Exception:
        return 'unknown'

## 1. Load data and split
`NORMALISE` lowercases text and removes punctuation. Keep it on unless Notebook 01 section 4 shows your corpus already contains capitals and punctuation. The backend must apply the same `normalise` function at prediction time.

In [ ]:
data_path = os.path.join(ANN_DIR, 'labelled_concerns.csv')
df = pd.read_csv(data_path)
assert {'sentence', 'label'}.issubset(df.columns)
assert df['sentence'].notna().all() and df['label'].notna().all()

DATA_ID = file_md5(data_path)[:8]
le = LabelEncoder()
y = le.fit_transform(df['label'])
CLASSES = le.classes_.tolist()
print(f'Dataset {DATA_ID}: {len(df):,} rows, classes {CLASSES}')

split_path = os.path.join(PROC_DIR, f'split_{DATA_ID}.npz')
train_idx, test_idx = train_test_split(
    np.arange(len(df)), test_size=TEST_SIZE, random_state=SEED, stratify=y)
np.savez(split_path, train_idx=train_idx, test_idx=test_idx)

print(f'Train {len(train_idx):,} | Test {len(test_idx):,} (held out)')
print(pd.Series(le.inverse_transform(y[train_idx])).value_counts())

## 2. AfroXLMR embeddings

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device, torch.cuda.get_device_name(0) if device == 'cuda' else '')

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME).to(device).eval()


def mean_pool(hidden, mask):
    mask = mask.unsqueeze(-1).float()
    return (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)


def embed(texts, batch_size=BATCH_SIZE):
    texts = [normalise(t) if NORMALISE else str(t) for t in texts]
    out = []
    for i in tqdm(range(0, len(texts), batch_size)):
        batch = tokenizer(texts[i:i + batch_size], return_tensors='pt', padding=True,
                          truncation=True, max_length=128).to(device)
        with torch.no_grad():
            hidden = model(**batch).last_hidden_state
        out.append(mean_pool(hidden, batch['attention_mask']).cpu().numpy())
    return np.vstack(out)

In [ ]:
emb_path = os.path.join(PROC_DIR, f'embeddings_{DATA_ID}_n{int(NORMALISE)}.npy')

if os.path.exists(emb_path):
    X_all = np.load(emb_path)
    print('Loaded cached embeddings')
else:
    X_all = embed(df['sentence'].tolist())
    np.save(emb_path, X_all)

assert X_all.shape == (len(df), 768)
assert np.isfinite(X_all).all()
print('Embeddings:', X_all.shape)

rt_path = os.path.join(ANN_DIR, 'resident_test.csv')
RES_ID, rt_emb_path = None, None

if os.path.exists(rt_path):
    rt = pd.read_csv(rt_path)
    assert set(rt['label']).issubset(CLASSES), 'resident_test.csv has unknown labels'
    RES_ID = file_md5(rt_path)[:8]
    rt_emb_path = os.path.join(PROC_DIR, f'resident_embeddings_{RES_ID}_n{int(NORMALISE)}.npy')
    if not os.path.exists(rt_emb_path):
        np.save(rt_emb_path, embed(rt['sentence'].tolist()))
    overlap = int(rt['sentence'].map(normalise).isin(df['sentence'].map(normalise)).sum())
    print(f'Resident test: {len(rt)} rows, {overlap} also in training data')
else:
    print('resident_test.csv not found. Create it before Notebook 03.')

## 3. Training partition and imbalance
Only the training partition is loaded. SMOTE is enabled only when the smallest class is under half the size of the largest.

In [ ]:
X_train, y_train = X_all[train_idx], y[train_idx]

counts = np.bincount(y_train)
ratio = counts.min() / counts.max()
USE_SMOTE = bool(ratio < 0.5)
print(f'Training class counts: {dict(zip(CLASSES, counts.tolist()))}')
print(f'Imbalance ratio {ratio:.2f} -> SMOTE inside CV: {USE_SMOTE}')

## 4. Tune and compare classifiers
All three use the same folds and the same metric. Tuning uses the training partition only.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
prefix = 'clf__' if USE_SMOTE else ''


def wrap(clf):
    if USE_SMOTE:
        return ImbPipeline([('smote', SMOTE(random_state=SEED)), ('clf', clf)])
    return clf


candidates = {
    'Logistic Regression': (
        wrap(LogisticRegression(max_iter=2000, class_weight='balanced', random_state=SEED)),
        {prefix + 'C': [0.1, 1, 10]},
        False),
    'Linear SVM': (
        wrap(CalibratedClassifierCV(
            LinearSVC(max_iter=5000, class_weight='balanced', random_state=SEED), cv=3)),
        {prefix + 'estimator__C': [0.1, 1, 10]},
        False),
    'XGBoost': (
        wrap(XGBClassifier(eval_metric='mlogloss', tree_method='hist', device=device,
                           random_state=SEED, verbosity=0)),
        {prefix + 'max_depth': [3, 5],
         prefix + 'learning_rate': [0.05, 0.1, 0.2],
         prefix + 'n_estimators': [100, 200],
         prefix + 'subsample': [0.8, 1.0]},
        True),
}

results = {}
for name, (estimator, grid, randomised) in candidates.items():
    start = time.time()
    if randomised:
        search = RandomizedSearchCV(estimator, grid, n_iter=10, cv=cv, scoring='f1_macro',
                                    random_state=SEED, n_jobs=1)
    else:
        search = GridSearchCV(estimator, grid, cv=cv, scoring='f1_macro', n_jobs=-1)
    search.fit(X_train, y_train)

    best = search.best_index_
    results[name] = {
        'search': search,
        'cv_f1': float(search.best_score_),
        'cv_std': float(search.cv_results_['std_test_score'][best]),
        'params': {k.replace('clf__', ''): v for k, v in search.best_params_.items()},
        'seconds': round(time.time() - start, 1),
    }
    print(f"{name}: CV macro F1 {results[name]['cv_f1']:.4f} +/- {results[name]['cv_std']:.4f} "
          f"({results[name]['seconds']}s) {results[name]['params']}")

## 5. Comparison and selection

In [ ]:
majority = cross_val_score(DummyClassifier(strategy='most_frequent'), X_train, y_train,
                           cv=cv, scoring='f1_macro')

table = pd.DataFrame([
    {'Model': n, 'CV macro F1': round(r['cv_f1'], 4), 'CV std': round(r['cv_std'], 4),
     'Seconds': r['seconds']} for n, r in results.items()
]).sort_values('CV macro F1', ascending=False).reset_index(drop=True)

print(table.to_string(index=False))
print(f'Majority-class baseline macro F1: {majority.mean():.4f}')

best_name = table.loc[0, 'Model']
top, top_std = table.loc[0, 'CV macro F1'], table.loc[0, 'CV std']
close = table[table['CV macro F1'] >= top - top_std]['Model'].tolist()
print(f'\nSelected: {best_name}')
print(f'Within one std of the best: {close}')
if len(close) > 1:
    print('These differences are not clearly meaningful. Report this honestly.')

plt.figure(figsize=(7, 3.5))
plt.barh(table['Model'], table['CV macro F1'], xerr=table['CV std'], color='#2563EB')
plt.axvline(majority.mean(), color='gray', linestyle='--', label='Majority baseline')
plt.xlabel('CV macro F1')
plt.legend()
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig(os.path.join(RES_DIR, 'model_comparison.png'), dpi=150)
plt.show()

## 6. Save models and metadata
The selected model and all three candidates are saved, along with everything needed to reproduce the run.

In [ ]:
slug = lambda n: n.lower().replace(' ', '_')

for name, r in results.items():
    joblib.dump(r['search'].best_estimator_, os.path.join(MODELS_DIR, slug(name) + '.pkl'))
    pd.DataFrame(r['search'].cv_results_).to_csv(
        os.path.join(RES_DIR, f'cv_results_{slug(name)}.csv'), index=False)

joblib.dump(results[best_name]['search'].best_estimator_, os.path.join(MODELS_DIR, 'selected_model.pkl'))
joblib.dump(le, os.path.join(MODELS_DIR, 'label_encoder.pkl'))

run_id = time.strftime('%Y%m%d_%H%M%S')
meta = {
    'run_id': run_id,
    'seed': SEED,
    'embedding_model': MODEL_NAME,
    'pooling': 'attention-mask mean',
    'normalise': NORMALISE,
    'dataset': {'file': 'labelled_concerns.csv', 'id': DATA_ID, 'rows': int(len(df)), 'classes': CLASSES},
    'split': {'file': os.path.basename(split_path), 'train': int(len(train_idx)),
              'test': int(len(test_idx)), 'test_size': TEST_SIZE},
    'embeddings_file': os.path.basename(emb_path),
    'resident_test': {'id': RES_ID, 'embeddings_file': os.path.basename(rt_emb_path) if rt_emb_path else None},
    'smote_in_cv': USE_SMOTE,
    'cv': {'folds': 5, 'metric': 'f1_macro'},
    'majority_baseline_macro_f1': round(float(majority.mean()), 4),
    'models': {n: {'cv_macro_f1': round(r['cv_f1'], 4), 'cv_std': round(r['cv_std'], 4),
                   'best_params': r['params']} for n, r in results.items()},
    'selected_model': best_name,
    'versions': {p: pkg_version(p) for p in ['torch', 'transformers', 'scikit-learn', 'xgboost', 'imbalanced-learn']},
}

encode = lambda o: o.item() if hasattr(o, 'item') else str(o)
for fname in ('experiment_metadata.json', f'experiment_{run_id}.json'):
    with open(os.path.join(RES_DIR, fname), 'w') as f:
        json.dump(meta, f, indent=2, default=encode)

print('Saved models to', MODELS_DIR)
print('Saved metadata to', RES_DIR)
print(f'Selected model: {best_name} | CV macro F1 {results[best_name]["cv_f1"]:.4f}')
print('Next: run Notebook 03 for held-out evaluation.')